# Compact BOI+CFO-shift FFT3 vector PT generation

This notebook has a single data source: `CSPB.ML_dataset`.  It does not read `fft3_basic_pattern_features_blind_v1.pkl`, previous `.pt` files, CSV caches, or any comparison file.

For each valid sample it computes the original BOI+CFO-shift pipeline and the three FFT branches internally, then saves only the compact final payload:
- `x_processed`: BOI+CFO-shifted processed signal, `complex64`, length 32768;
- `vectors["lr445"]`, `vectors["lr55"]`, `vectors["lr7"]`;
- `vectors["rs18"]`, `vectors["rs28"]`, `vectors["rs20"]`, and `vectors["cfo1"] == [x4_top1_f]`;
- truth Rs/CFO/SNR plus `cfo_truth_shifted`;
- BOI shift/filter metadata;
- BOI power fields, with `p_signal_boi_local` renamed to `p_boi_inband_signal`.

2018 negative-truth-SNR samples are skipped before BOI/FFT and recorded in each batch pack's `errors` list. No CSV files are written.


In [1]:
from pathlib import Path
import sys


def _find_helper_project_dir(start=Path.cwd()):
    start = Path(start).resolve()
    search_roots = [start, *start.parents]
    for root in search_roots:
        if (root / "helper_function" / "__init__.py").exists():
            return root
    for root in search_roots[:3]:
        matches = list(root.glob("**/boi_shift_fft3_cfpattern_spl/helper_function/__init__.py"))
        if matches:
            return matches[0].parent.parent
    raise FileNotFoundError("Could not find helper_function from the current Jupyter working directory.")


PROJECT_DIR = _find_helper_project_dir()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from helper_function.pre_compact_fft3_vector_generator import (
    LR7_FEATURES,
    LR55_FEATURES,
    RS18_FEATURES,
    RS28_FEATURES,
    RS20_FEATURES,
    generate_compact_vector_pt_files,
    inspect_compact_pt,
)

BASE = PROJECT_DIR
OUTPUT_ROOT = BASE

print("BASE:", BASE)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("No pkl/cache input is used.")


BASE: /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode
OUTPUT_ROOT: /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode
No pkl/cache input is used.


## Fixed Feature Sets

The 7/55 LR vectors and 18/28/20 Rs vectors are fixed lists.  The 445-vector feature order is derived from the first valid sample using the same blind feature construction logic, then stored once per batch in `vector_feature_names["lr445"]`.


In [2]:
print("lr7 count:", len(LR7_FEATURES))
print("lr55 count:", len(LR55_FEATURES))
print("rs18 count:", len(RS18_FEATURES), "first:", RS18_FEATURES[0])
print("rs28 count:", len(RS28_FEATURES), "first:", RS28_FEATURES[0])
print("rs20 count:", len(RS20_FEATURES), "first:", RS20_FEATURES[0])

assert len(LR7_FEATURES) == 7
assert len(LR55_FEATURES) == 55
assert len(RS18_FEATURES) == 18
assert len(RS28_FEATURES) == 28
assert len(RS20_FEATURES) == 20
assert RS18_FEATURES[0] == "x4_top1_f"
assert RS28_FEATURES[0] == "x4_top1_f"
assert RS20_FEATURES[0] == "abs_x2_top1_minus_top2_db"


lr7 count: 7
lr55 count: 55
rs18 count: 18 first: x4_top1_f
rs28 count: 28 first: x4_top1_f
rs20 count: 20 first: abs_x2_top1_minus_top2_db


In [3]:
print("LR7_FEATURES =")
for i, name in enumerate(LR7_FEATURES, 1):
    print(f"{i:02d}. {name}")

print("\nRS18_FEATURES =")
for i, name in enumerate(RS18_FEATURES, 1):
    print(f"{i:02d}. {name}")

print("\nRS28_FEATURES =")
for i, name in enumerate(RS28_FEATURES, 1):
    print(f"{i:02d}. {name}")


LR7_FEATURES =
01. x4_top2_db
02. x4_top1_af
03. x4_top1_db
04. x4_top1_minus_top5_db
05. x2_top4_db
06. x2_top1_db
07. abs_x2_top1_minus_top5_db

RS18_FEATURES =
01. x4_top1_f
02. x4_top2_f
03. x4_top3_f
04. x4_top1_db
05. x4_top2_db
06. x4_top3_db
07. abs_x2_top2_af
08. abs_x2_top3_af
09. abs_x2_top4_af
10. abs_x2_top5_af
11. abs_x2_top6_af
12. abs_x2_top7_af
13. abs_x2_top8_af
14. abs_x2_top10_af
15. abs_x2_top12_af
16. abs_x2_top14_af
17. abs_x2_top16_af
18. rs_est

RS28_FEATURES =
01. x4_top1_f
02. x4_top2_f
03. x4_top3_f
04. x4_top4_f
05. x4_top1_db
06. x4_top2_db
07. x4_top3_db
08. x4_top4_db
09. abs_x2_top2_af
10. abs_x2_top3_af
11. abs_x2_top4_af
12. abs_x2_top5_af
13. abs_x2_top6_af
14. abs_x2_top7_af
15. abs_x2_top8_af
16. abs_x2_top9_af
17. abs_x2_top10_af
18. abs_x2_top11_af
19. abs_x2_top12_af
20. abs_x2_top13_af
21. abs_x2_top14_af
22. abs_x2_top15_af
23. abs_x2_top16_af
24. abs_x2_top17_af
25. abs_x2_top18_af
26. abs_x2_top19_af
27. abs_x2_top20_af
28. rs_est


## Generation Config

Default output is this folder: `2018/` and `2022/` will be created under `BASE`, producing 56 compact batch `.pt` files.


In [4]:
CFG = {
    "ROOT_MAP": {
        "2022": str((BASE / "../../CSPB.ML_dataset/CSPB_ML_2022_Data").resolve()),
        "2018": str((BASE / "../../CSPB.ML_dataset/CSPB_ML_2018_Data").resolve()),
    },
    "LABEL_MAP": {
        "2022": "CSPB_ML_2022_Signal_Truth_Labels.txt",
        "2018": "CSPB_ML_2018_Signal_Truth_Labels.txt",
    },
    "N_PER_BATCH": 4000,
    "N_BATCHES": 28,
    "N_TOTAL": 4000 * 28,
    "SAMPLES_PER_SIG": 32768,
    "FS_IN": 1.0,
    "EPS": 1e-12,
    "OUT_ROOT": str(OUTPUT_ROOT),
    "SKIP_EXISTING": False,
    "DATASET_SELECT": "2022",
    "START_PT": 1,
    "END_PT": 28,
    "DELTA_F0": 0.002,
    "THRESHOLD_DB": 5.0,
    "BOUNDARY_THRESHOLD_DB": 1.5,
    "BOI_SELECT_MODE": "strongest",
    "MERGE_GAP_W_FACTOR": 2.0,
    "MAX_EXPAND_W_FACTOR": 8.0,
    "BOI_GUARD_BINS": 2,
    "ADAPTIVE_FILTER_EXPAND": True,
    "FILTER_EXPAND_RATIO": 0.20,
    "FILTER_EXPAND_RATIO_LOW_SNR": 0.30,
    "FILTER_EXPAND_RATIO_MID_SNR": 0.15,
    "FILTER_EXPAND_RATIO_HIGH_SNR": 0.05,
    "SNR_LOW_TH_DB": 4.0,
    "SNR_HIGH_TH_DB": 8.0,
    "MAX_EXPANDED_FILTER_BW_FRAC": 0.95,
    "FFT_NFFT": 32768,
    "FFT_WINDOW": "hann",
    "FFT_TOP_K": 20,
    "FFT_PEAK_MIN_SEP_BINS": 8,
    "GUIDE_KMAX_INTEGER_RS": 5,
    "GUIDE_KMAX_HALF_RS": 5,
    "GUIDE_SEARCH_BINS": 3,
}

print("Will generate to:", CFG["OUT_ROOT"])
print("Dataset select:", CFG["DATASET_SELECT"], "batches:", CFG["START_PT"], "to", CFG["END_PT"])


Will generate to: /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode
Dataset select: 2022 batches: 1 to 28


## Run Generation

This cell performs the full 2018/2022 generation from `CSPB.ML_dataset`.  It does not save CSV files; skip/error records are kept inside each compact batch `.pt`.


In [5]:
generate_compact_vector_pt_files(CFG)


Compact BOI + Shift + FFT3 vector PT generation
dataset_name     : 2022
label_path       : /home/ruilin/SPL_Surreal/CSPB.ML_dataset/CSPB_ML_2022_Data/CSPB_ML_2022_Signal_Truth_Labels.txt
root_data        : /home/ruilin/SPL_Surreal/CSPB.ML_dataset/CSPB_ML_2022_Data
out_dir          : /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode/2022
nfft             : 32768
window           : hann
Start 2022 batch 01
Signal index range: 1 to 4000
Output: /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode/2022/2022_batch_01_boi_shift_fft3_compact_vectors.pt


2022 compact batch 01:   0%|          | 0/4000 [00:00<?, ?it/s]

Saved 2022 compact batch 01: samples=4000 | errors=0
Start 2022 batch 02
Signal index range: 4001 to 8000
Output: /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode/2022/2022_batch_02_boi_shift_fft3_compact_vectors.pt


2022 compact batch 02:   0%|          | 0/4000 [00:00<?, ?it/s]

Saved 2022 compact batch 02: samples=4000 | errors=0
Start 2022 batch 03
Signal index range: 8001 to 12000
Output: /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode/2022/2022_batch_03_boi_shift_fft3_compact_vectors.pt


...

Saved 2022 compact batch 26: samples=4000 | errors=0
Start 2022 batch 27
Signal index range: 104001 to 108000
Output: /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode/2022/2022_batch_27_boi_shift_fft3_compact_vectors.pt


2022 compact batch 27:   0%|          | 0/4000 [00:00<?, ?it/s]

Saved 2022 compact batch 27: samples=4000 | errors=0
Start 2022 batch 28
Signal index range: 108001 to 112000
Output: /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode/2022/2022_batch_28_boi_shift_fft3_compact_vectors.pt


2022 compact batch 28:   0%|          | 0/4000 [00:00<?, ?it/s]

Saved 2022 compact batch 28: samples=4000 | errors=0
Done. Compact PT files saved under: /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode
Errors are stored inside each .pt batch only; no CSV was written. total_errors=0


## Inspect One Output File

Run after generation. The inspected sample should have no `fft` key; vectors should have lengths 445, 55, 7, 18, 28, 20, and 1.


In [6]:
example_pt = OUTPUT_ROOT / "2018" / "2018_batch_01_boi_shift_fft3_compact_vectors.pt"
inspect_compact_pt(example_pt)


{'path': '/home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode/2018/2018_batch_01_boi_shift_fft3_compact_vectors.pt',
 'version': 'boi_shift_fft3_compact_vectors_batch_v1',
 'num_samples_saved': 3992,
 'num_errors': 8,
 'sample_keys': ['version',
  'meta',
  'truth',
  'preprocess',
  'power',
  'fft_config',
  'x_processed',
  'vectors',
  'summary'],
 'has_fft_key': False,
 'x_processed_shape': (32768,),
 'x_processed_dtype': 'complex64',
 'vector_shapes': {'lr445': (445,),
  'lr55': (55,),
  'lr7': (7,),
  'rs18': (18,),
  'rs28': (28,),
  'rs20': (20,),
  'cfo1': (1,)},
 'power_keys': ['P_total',
  'P_x_bpf',
  'P_x_processed',
  'p_noise_inband_hat',
  'snr_recommended_db',
  'p_boi_inband_signal']}